# Dallas Housing Needs Assessment

Estimating Dallas households below 60% of AMI, identifying the low-income cost-burdened population, and summarizing implications for housing policy.


## Assessment Questions

This notebook is organized around the three assessment questions: scale of need, descriptive indicators of burden, and policy implications.

1. How many households fall below 60% of AMI, and how does that differ by tenure?
2. Which household characteristics are most associated descriptively with housing cost burden among low-income households?
3. What policy implications follow from those results?


In [123]:
# %pip install pandas matplotlib openpyxl requests altair vl-convert-python
from pathlib import Path
import textwrap
import altair as alt
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import numpy as np
import requests
from IPython.display import Markdown, display
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / '230301_assessment_data_2.xlsx'
GLOSSARY_PATH = PROJECT_ROOT / 'data' / '230301_assessment_glossary_2.xlsx'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
TABLE_DIR = OUTPUT_DIR / 'tables'
CHART_DIR = OUTPUT_DIR / 'charts'
REPORT_DIR = OUTPUT_DIR / 'report'
for path in [TABLE_DIR, CHART_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)
ACCENT = '#2E5B78'
ACCENT_LIGHT = '#9FB7C8'
DARK = '#1F2D38'
MID = '#63727D'
LIGHT = '#DCE5EC'
GRID = '#E6EBEF'
BG = '#F7F5F0'
LINE = '#CCD5DC'
plt.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor': BG,
    'savefig.facecolor': BG,
    'font.size': 11,
    'font.family': 'sans-serif',
    'font.sans-serif': ['IBM Plex Sans', 'DejaVu Sans', 'Arial', 'Liberation Sans'],
    'axes.titlesize': 18,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.edgecolor': LINE,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.color': MID,
    'ytick.color': MID,
    'axes.labelcolor': MID,
    'text.color': DARK,
})
alt.data_transformers.disable_max_rows()

ALT_FONT = 'IBM Plex Sans'


## 1. Inputs

Confirming the local workbooks, previews the supplied data and glossary, and checks key raw coding decisions before analysis.


In [124]:
data_book = pd.ExcelFile(DATA_PATH)
glossary_book = pd.ExcelFile(GLOSSARY_PATH)
data_sheet_names = data_book.sheet_names
glossary_sheet_names = glossary_book.sheet_names
display(pd.DataFrame({
    'workbook': ['assessment data', 'assessment glossary'],
    'sheet_names': [', '.join(data_sheet_names), ', '.join(glossary_sheet_names)]
}))
data_raw = pd.read_excel(DATA_PATH, sheet_name=data_sheet_names[0])
glossary_raw = pd.read_excel(GLOSSARY_PATH, sheet_name=glossary_sheet_names[0])
display(data_raw.head())
display(glossary_raw.head(15))
variable_guide = pd.DataFrame([
    {'concept': 'Household income', 'analysis_variable': 'HH_Income', 'raw_source': 'HINCP + ADJINC', 'notes': 'Inflation-adjusted household income. Used for AMI classification and burden denominator.'},
    {'concept': 'Household size', 'analysis_variable': 'NP', 'raw_source': 'NP', 'notes': 'Number of people in household; mapped to 60% AMI thresholds.'},
    {'concept': 'Tenure', 'analysis_variable': 'Tenure', 'raw_source': 'TEN', 'notes': 'Cleaned owner vs renter field. Raw TEN codes follow ACS PUMS and are collapsed into owner / renter categories.'},
    {'concept': 'Household weight', 'analysis_variable': 'WGTP', 'raw_source': 'WGTP', 'notes': 'Household weight used in all estimates.'},
    {'concept': 'Renter monthly housing cost', 'analysis_variable': 'Rent', 'raw_source': 'GRNTP + ADJHSG', 'notes': 'Inflation-adjusted monthly gross rent for renter households.'},
    {'concept': 'Owner monthly housing cost', 'analysis_variable': 'Owner_Costs', 'raw_source': 'SMOCP + ADJHSG', 'notes': 'Inflation-adjusted selected monthly owner costs.'},
    {'concept': 'Race / ethnicity', 'analysis_variable': 'RACE', 'raw_source': 'RAC1P + HISP', 'notes': 'Cleaned race / ethnicity grouping supplied in the local file; best interpreted as the household reference person / householder.'},
    {'concept': 'Children / family structure', 'analysis_variable': 'Children_Exist, Single_Parent_HH', 'raw_source': 'Derived', 'notes': 'Useful for vulnerability profiling and next-step modeling.'},
    {'concept': 'Senior presence', 'analysis_variable': 'Seniors_Exist', 'raw_source': 'Derived', 'notes': 'Useful age-related proxy at the household level.'},
    {'concept': 'Housing type', 'analysis_variable': 'typology', 'raw_source': 'BLD derived', 'notes': 'Contextual housing form variable.'},
    {'concept': 'Recent mobility', 'analysis_variable': 'Moved', 'raw_source': 'MIG derived', 'notes': 'Flags households that moved within the last 24 months.'},
])
display(variable_guide)


,workbook,sheet_names
0,assessment data,230301_assessment_data (2)
1,assessment glossary,230301_assessment_glossary (2)


,NP,WGTP,SERIALNO,TEN,HINCP,BDSP,GRNTP,BLD,HHT,SMOCP,...,Children,col_students,Young Adults,Working_Age_Adults,Single_Parent_HH,Seniors_Only,Seniors_Exist,Children_Exist,typology,Moved
0,1,40,2017000000000,3,70000,0,1055.0,9,4,NaN,...,0,0,1,1,0,0,0,0,50+ Units,1
1,1,14,2017000000000,3,58000,0,1080.0,9,6,NaN,...,0,0,1,1,0,0,0,0,50+ Units,1
2,1,11,2017000000000,3,25000,0,420.0,7,4,NaN,...,0,0,0,1,0,0,0,0,10-49 Units,0
3,1,23,2017000000000,3,16500,0,650.0,8,4,NaN,...,0,0,1,1,0,0,0,0,10-49 Units,1
4,1,17,2017000000000,3,105000,0,860.0,5,4,NaN,...,0,0,0,1,0,0,0,0,2-9 Units,1


,Variable,Definition,Unnamed: 2
0,NP,Original Variable,https://www2.census.gov/programs-surveys/acs/t...
1,WGTP,Original Variable,NaN
2,SERIALNO,Original Variable,NaN
3,TEN,Original Variable,NaN
4,HINCP,Original Variable,NaN
5,BDSP,Original Variable,NaN
6,GRNTP,Original Variable,NaN
7,BLD,Original Variable,NaN
8,HHT,Original Variable,NaN
9,SMOCP,Original Variable,NaN


,concept,analysis_variable,raw_source,notes
0,Household income,HH_Income,HINCP + ADJINC,Inflation-adjusted household income. Used for ...
1,Household size,NP,NP,Number of people in household; mapped to 60% A...
2,Tenure,Tenure,TEN,Cleaned owner vs renter field. Raw TEN codes f...
3,Household weight,WGTP,WGTP,Household weight used in all estimates.
4,Renter monthly housing cost,Rent,GRNTP + ADJHSG,Inflation-adjusted monthly gross rent for rent...
5,Owner monthly housing cost,Owner_Costs,SMOCP + ADJHSG,Inflation-adjusted selected monthly owner costs.
6,Race / ethnicity,RACE,RAC1P + HISP,Cleaned race / ethnicity grouping supplied in ...
7,Children / family structure,"Children_Exist, Single_Parent_HH",Derived,Useful for vulnerability profiling and next-st...
8,Senior presence,Seniors_Exist,Derived,Useful age-related proxy at the household level.
9,Housing type,typology,BLD derived,Contextual housing form variable.


## 2. Variable scope

Narrowing the supplied fields to a concise set of primary analysis variables, validation checks, and optional extensions.


In [125]:
analysis_scope_guide = pd.DataFrame([
    {'variable': 'HH_Income', 'use_in_notebook': 'Primary', 'why': 'Core denominator for AMI classification and housing-cost ratio.'},
    {'variable': 'Rent', 'use_in_notebook': 'Primary', 'why': 'Primary renter housing cost input.'},
    {'variable': 'Owner_Costs', 'use_in_notebook': 'Primary', 'why': 'Primary owner housing cost input.'},
    {'variable': 'Electric_Costs', 'use_in_notebook': 'Not primary', 'why': 'Useful context, but excluded from the core burden formula to avoid double counting if already embedded in adjusted housing costs.'},
    {'variable': 'Public_Assistance', 'use_in_notebook': 'Secondary', 'why': 'Potential hardship marker, but not central to answering the assessment questions.'},
    {'variable': 'Public_Assistance_Flag', 'use_in_notebook': 'Secondary', 'why': 'Useful as a descriptive vulnerability flag, but not a top-tier indicator for the timed assessment.'},
    {'variable': 'cost_burden_rate', 'use_in_notebook': 'Validation only', 'why': 'Supplied derived field used to check consistency against the notebook’s own burden calculation.'},
    {'variable': 'cost_burden', 'use_in_notebook': 'Validation only', 'why': 'Supplied boolean burden field used as a cross-check rather than as the main analysis variable.'},
    {'variable': 'severe_cost_burden', 'use_in_notebook': 'Optional follow-up', 'why': 'Useful extension, but not required to answer the prompt.'},
    {'variable': 'Tenure', 'use_in_notebook': 'Primary', 'why': 'Strongest and most policy-relevant split in the burden analysis.'},
    {'variable': 'RACE', 'use_in_notebook': 'Primary', 'why': 'Important for demographic comparison and equity framing.'},
    {'variable': 'Seniors / Seniors_Exist / Seniors_Only', 'use_in_notebook': 'Selected representation', 'why': 'Use one senior-oriented measure rather than several overlapping versions.'},
    {'variable': 'Adults / Children', 'use_in_notebook': 'Derived grouping support', 'why': 'Helpful for household composition, but cleaner to summarize through grouped size or family indicators.'},
    {'variable': 'col_students', 'use_in_notebook': 'Not primary', 'why': 'Interesting but not essential for a concise housing-policy story.'},
    {'variable': 'Single_Parent_HH', 'use_in_notebook': 'Primary candidate', 'why': 'Strong family vulnerability indicator.'},
    {'variable': 'Children_Exist', 'use_in_notebook': 'Primary candidate', 'why': 'Useful family presence flag without overcomplicating household composition.'},
    {'variable': 'typology', 'use_in_notebook': 'Primary candidate', 'why': 'Adds policy-relevant housing context and performs well descriptively.'},
    {'variable': 'Moved', 'use_in_notebook': 'Primary candidate', 'why': 'Captures recent exposure to current market conditions.'},
])
display(Markdown('### Selected variables'))
display(analysis_scope_guide)
display(Markdown('Variables were narrowed to a short set of policy-relevant indicators; overlapping fields and supplied outcome variables are retained only for checks or future extensions.'))
tenure_coding = (
    data_raw[['TEN', 'Tenure']]
    .drop_duplicates()
    .sort_values(['TEN', 'Tenure'])
    .rename(columns={'TEN': 'raw_code', 'Tenure': 'clean_label'})
)
race_coding_preview = (
    data_raw[['RAC1P', 'HISP', 'RACE']]
    .value_counts()
    .reset_index(name='rows')
    .sort_values('rows', ascending=False)
    .head(12)
)
serial_counts = data_raw['SERIALNO'].value_counts(dropna=False)
serial_issue_summary = pd.DataFrame([
    {
        'rows_in_file': len(data_raw),
        'unique_serialno_values': int(data_raw['SERIALNO'].nunique(dropna=True)),
        'rows_with_repeated_serialno': int(serial_counts[serial_counts > 1].sum()),
        'largest_repeat_count_for_one_serialno': int(serial_counts.max()),
    }
])
coding_audit = pd.DataFrame([
    {'concept': 'Tenure raw coding', 'detail': 'Observed TEN mapping in local file: 1 and 2 map to owners; 3 and 4 map to renters.'},
    {'concept': 'Race raw coding', 'detail': 'Observed RAC1P codes in the file include 1, 2, 3, 5, 6, 7, 8, 9. The cleaned RACE field collapses these and also absorbs Hispanic origin into a Hispanic category.'},
    {'concept': 'Hispanic / Latino coding', 'detail': 'Observed HISP coding follows ACS structure: HISP = 1 indicates non-Hispanic; values greater than 1 indicate Hispanic / Latino origin groups.'},
    {'concept': 'Weight and costs', 'detail': 'WGTP is the household weight. HH_Income, Rent, and Owner_Costs are already inflation-adjusted fields in the supplied file.'},
])
display(Markdown('### Raw coding checks for key ACS variables'))
display(tenure_coding)
display(race_coding_preview)
display(coding_audit)
display(serial_issue_summary)
display(Markdown('`SERIALNO` is not reliable as a unique household identifier in the supplied extract, so it is not used for deduplication or household-person linkage.'))


### Selected variables

,variable,use_in_notebook,why
0,HH_Income,Primary,Core denominator for AMI classification and ho...
1,Rent,Primary,Primary renter housing cost input.
2,Owner_Costs,Primary,Primary owner housing cost input.
3,Electric_Costs,Not primary,"Useful context, but excluded from the core bur..."
4,Public_Assistance,Secondary,"Potential hardship marker, but not central to ..."
5,Public_Assistance_Flag,Secondary,"Useful as a descriptive vulnerability flag, bu..."
6,cost_burden_rate,Validation only,Supplied derived field used to check consisten...
7,cost_burden,Validation only,Supplied boolean burden field used as a cross-...
8,severe_cost_burden,Optional follow-up,"Useful extension, but not required to answer t..."
9,Tenure,Primary,Strongest and most policy-relevant split in th...


Variables were narrowed to a short set of policy-relevant indicators; overlapping fields and supplied outcome variables are retained only for checks or future extensions.

### Raw coding checks for key ACS variables

,raw_code,clean_label
13,1,Owner Household
46,2,Owner Household
0,3,Renter Household
61,4,Renter Household


,RAC1P,HISP,RACE,rows
0,1,1,White,11853
1,2,1,Black,4537
2,1,2,Hispanic,3312
3,8,2,Hispanic,1090
4,6,1,Asian,866
5,9,2,Hispanic,795
6,9,1,Other,418
7,1,11,Hispanic,121
8,1,24,Hispanic,112
9,8,11,Hispanic,76


,concept,detail
0,Tenure raw coding,Observed TEN mapping in local file: 1 and 2 ma...
1,Race raw coding,"Observed RAC1P codes in the file include 1, 2,..."
2,Hispanic / Latino coding,Observed HISP coding follows ACS structure: HI...
3,Weight and costs,"WGTP is the household weight. HH_Income, Rent,..."


,rows_in_file,unique_serialno_values,rows_with_repeated_serialno,largest_repeat_count_for_one_serialno
0,24243,18903,5341,5341


`SERIALNO` is not reliable as a unique household identifier in the supplied extract, so it is not used for deduplication or household-person linkage.

## 3. Workflow

Defining the reusable functions for loading data, classifying AMI status, calculating burden, building summaries, and producing charts.


In [126]:
# Dallas 60% AMI thresholds derived from the Dallas 2020 HUD income limits/.
# These are a Dallas-specific, earlier-vintage benchmark relative to the ACS / PUMS data.
DALLAS_AMI_60_LIMITS_2020 = {
    1: 36240,
    2: 41400,
    3: 46560,
    4: 51720,
    5: 55860,
    6: 60000,
    7: 64140,
    8: 68280,
    9: 69840,
}
DALLAS_PUMAS = ['02304', '02305', '02306', '02307', '02309', '02311', '02312', '02313', '02314', '02315', '02316', '02319']
REQUIRED_COLUMNS = [
    'WGTP', 'NP', 'HH_Income', 'Tenure', 'Rent', 'Owner_Costs', 'RACE',
    'Children_Exist', 'Single_Parent_HH', 'Seniors_Exist', 'typology', 'Moved', 'PUMA'
]
CANDIDATE_VARIABLES = {
    'Tenure': {
        'label': 'Tenure',
        'level': 'Household',
        'why_it_matters': 'Tenure captures different affordability mechanisms for renters and owners and is usually the strongest descriptive split for cost burden.',
        'summary_method': 'Weighted counts and burden rates by renter versus owner.',
        'model_ready': 'Yes'
    },
    'race_ethnicity': {
        'label': 'Race / ethnicity of reference person',
        'level': 'Reference person / household proxy',
        'why_it_matters': 'Racial and ethnic disparities often reflect long-run inequities in housing access, labor markets, and wealth accumulation.',
        'summary_method': 'Weighted population shares for all households versus the target group.',
        'model_ready': 'Yes'
    },
    'household_size_group': {
        'label': 'Household size',
        'level': 'Household',
        'why_it_matters': 'Household size affects both AMI thresholds and housing consumption needs.',
        'summary_method': 'Weighted shares and burden rates by size band.',
        'model_ready': 'Yes'
    },
    'Single_Parent_HH': {
        'label': 'Single-parent household',
        'level': 'Household',
        'why_it_matters': 'Single-parent households often have tighter budgets and stronger demand for stable, family-sized affordable units.',
        'summary_method': 'Burden rates within the below-60%-AMI subset, paired with target-group composition.',
        'model_ready': 'Yes'
    },
    'Seniors_Exist': {
        'label': 'Senior present',
        'level': 'Household',
        'why_it_matters': 'Senior households may face fixed-income constraints and distinct policy needs such as tax relief, accessibility, or preservation.',
        'summary_method': 'Burden rates and target-group shares.',
        'model_ready': 'Yes'
    },
    'typology': {
        'label': 'Structure type / housing typology',
        'level': 'Household',
        'why_it_matters': 'Housing form provides context for where affordability pressure appears to be concentrated.',
        'summary_method': 'Weighted shares and burden rates by typology.',
        'model_ready': 'Yes'
    },
    'Moved': {
        'label': 'Moved in last 24 months',
        'level': 'Household',
        'why_it_matters': 'Recent movers may be more exposed to current market rents and weaker housing stability.',
        'summary_method': 'Burden rates and target-group shares by recent mobility.',
        'model_ready': 'Yes'
    },
}
def load_input_data(data_path=DATA_PATH, glossary_path=GLOSSARY_PATH):
    data_book = pd.ExcelFile(data_path)
    glossary_book = pd.ExcelFile(glossary_path)
    data = pd.read_excel(data_path, sheet_name=data_book.sheet_names[0])
    glossary = pd.read_excel(glossary_path, sheet_name=glossary_book.sheet_names[0])
    return {
        'data_book': data_book,
        'glossary_book': glossary_book,
        'data': data,
        'glossary': glossary,
    }
def inspect_and_standardize_columns(df):
    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    numeric_cols = ['WGTP', 'NP', 'HH_Income', 'Rent', 'Owner_Costs', 'Single_Parent_HH', 'Children_Exist', 'Seniors_Exist', 'Moved']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['PUMA'] = df['PUMA'].astype('Int64').astype(str).str.zfill(5)
    df['puma5'] = '48' + df['PUMA']
    return df
def assign_ami_thresholds(df, ami_limits=DALLAS_AMI_60_LIMITS_2020):
    df = df.copy()
    df['household_size_for_ami'] = df['NP'].clip(lower=1, upper=9).fillna(1).astype(int)
    df['ami_60_threshold'] = df['household_size_for_ami'].map(ami_limits)
    df['below_60_ami'] = df['HH_Income'] < df['ami_60_threshold']
    return df
def calculate_housing_cost_burden(df):
    df = df.copy()
    df['monthly_housing_cost'] = pd.Series(index=df.index, dtype='float64')
    renter_mask = df['Tenure'].eq('Renter Household')
    owner_mask = df['Tenure'].eq('Owner Household')
    df.loc[renter_mask, 'monthly_housing_cost'] = df.loc[renter_mask, 'Rent']
    df.loc[owner_mask, 'monthly_housing_cost'] = df.loc[owner_mask, 'Owner_Costs']
    df['annual_housing_cost'] = df['monthly_housing_cost'] * 12
    df['non_positive_income_flag'] = df['HH_Income'].fillna(0) <= 0
    valid_ratio = (~df['non_positive_income_flag']) & df['annual_housing_cost'].notna()
    df['housing_cost_ratio'] = pd.NA
    df.loc[valid_ratio, 'housing_cost_ratio'] = (
        df.loc[valid_ratio, 'annual_housing_cost'] / df.loc[valid_ratio, 'HH_Income']
    )
    df['cost_burdened'] = df['housing_cost_ratio'] > 0.30
    df['cost_burdened'] = df['cost_burdened'].fillna(False)
    df['target_group'] = df['below_60_ami'] & df['cost_burdened']
    return df
def add_analysis_fields(df):
    df = df.copy()
    df['tenure_simple'] = df['Tenure'].replace({
        'Renter Household': 'Renter',
        'Owner Household': 'Owner'
    })
    df['ami_group'] = df['below_60_ami'].map({True: 'Below 60% AMI', False: 'At/Above 60% AMI'})
    df['household_size_group'] = pd.cut(
        df['NP'],
        bins=[0, 1, 2, 4, 100],
        labels=['1 person', '2 persons', '3-4 persons', '5+ persons'],
        include_lowest=True,
    )
    df['single_parent_label'] = df['Single_Parent_HH'].map({1: 'Single-parent', 0: 'Other households'})
    df['children_label'] = df['Children_Exist'].map({1: 'Children present', 0: 'No children'})
    df['senior_label'] = df['Seniors_Exist'].map({1: 'Senior present', 0: 'No senior'})
    df['moved_label'] = df['Moved'].map({1: 'Moved in last 24 months', 0: 'Did not move recently'})
    df['race_ethnicity'] = df['RACE']
    return df
def build_weighted_summary(df, group_cols, weight_col='WGTP', value_name='weighted_households'):
    return (
        df.groupby(group_cols, dropna=False)[weight_col]
        .sum()
        .reset_index(name=value_name)
        .sort_values(value_name, ascending=False)
    )
def summarize_distribution(df, variable, group_flag='target_group', weight_col='WGTP'):
    all_dist = build_weighted_summary(df, [variable], weight_col=weight_col).rename(columns={variable: 'category'})
    target_dist = build_weighted_summary(df[df[group_flag]], [variable], weight_col=weight_col).rename(columns={variable: 'category'})
    all_total = all_dist['weighted_households'].sum()
    target_total = target_dist['weighted_households'].sum()
    summary = all_dist.merge(target_dist, on='category', how='outer', suffixes=('_all', '_target')).fillna(0)
    summary['share_all'] = summary['weighted_households_all'] / all_total
    summary['share_target'] = summary['weighted_households_target'] / target_total
    summary['share_point_difference'] = summary['share_target'] - summary['share_all']
    return summary.sort_values('share_target', ascending=False)
def burden_rate_within_subset(df, variable, subset_mask, weight_col='WGTP'):
    subset = df.loc[subset_mask & (~df['non_positive_income_flag']) & df['annual_housing_cost'].notna()].copy()
    grouped = subset.groupby(variable, dropna=False).apply(
        lambda g: pd.Series({
            'weighted_households': g[weight_col].sum(),
            'weighted_burdened': g.loc[g['cost_burdened'], weight_col].sum(),
        })
    ).reset_index()
    grouped['burden_rate'] = grouped['weighted_burdened'] / grouped['weighted_households']
    return grouped.sort_values('burden_rate', ascending=False)
def compare_target_group(df):
    selected = {
        'Tenure': 'Tenure',
        'race_ethnicity': 'Race / ethnicity of reference person',
        'household_size_group': 'Household size',
        'single_parent_label': 'Single-parent household',
        'senior_label': 'Senior present',
        'typology': 'Housing typology',
        'moved_label': 'Moved in last 24 months',
    }
    outputs = []
    for variable, label in selected.items():
        summary = summarize_distribution(df, variable)
        summary.insert(0, 'variable', label)
        outputs.append(summary)
    return pd.concat(outputs, ignore_index=True)
def build_validation_check(df):
    if not {'cost_burden_rate', 'cost_burden'}.issubset(df.columns):
        return pd.DataFrame()
    validation = df.copy()
    validation['supplied_cost_burden_rate'] = pd.to_numeric(validation['cost_burden_rate'], errors='coerce')
    validation['supplied_cost_burden'] = pd.to_numeric(validation['cost_burden'], errors='coerce')
    compare_mask = validation['housing_cost_ratio'].notna() & validation['supplied_cost_burden_rate'].notna()
    burden_mask = validation['supplied_cost_burden'].notna()
    summary = pd.DataFrame([
        {
            'rows_compared_on_burden_rate': int(compare_mask.sum()),
            'mean_abs_difference_pct_points': ((validation.loc[compare_mask, 'housing_cost_ratio'] * 100 - validation.loc[compare_mask, 'supplied_cost_burden_rate']).abs().mean()),
            'rows_compared_on_burden_flag': int((validation['housing_cost_ratio'].notna() & burden_mask).sum()),
            'burden_flag_agreement_rate': (
                (validation.loc[validation['housing_cost_ratio'].notna() & burden_mask, 'cost_burdened'].astype(int) == validation.loc[validation['housing_cost_ratio'].notna() & burden_mask, 'supplied_cost_burden'].astype(int)).mean()
            ),
        }
    ])
    return summary
def identify_indicator_variable(df):
    candidates = {
        'Tenure': 'Tenure',
        'single_parent_label': 'Single-parent household',
        'household_size_group': 'Household size',
        'senior_label': 'Senior presence',
        'moved_label': 'Recent move status',
        'typology': 'Housing typology',
    }
    rows = []
    subset = df['below_60_ami']
    for variable, label in candidates.items():
        summary = burden_rate_within_subset(df, variable, subset)
        spread = summary['burden_rate'].max() - summary['burden_rate'].min()
        rows.append({
            'variable': variable,
            'label': label,
            'spread': spread,
            'use_for_chart_3': variable != 'Tenure',
        })
    ranking = pd.DataFrame(rows).sort_values('spread', ascending=False)
    return ranking
def select_chart_3_variable(indicator_ranking):
    chart_candidates = indicator_ranking.loc[indicator_ranking['use_for_chart_3']].copy()
    if chart_candidates.empty:
        return {'variable': 'Tenure', 'label': 'Tenure'}
    top = chart_candidates.iloc[0]
    return {'variable': top['variable'], 'label': top['label']}
def format_households(value):
    if pd.isna(value):
        return ''
    return f'{value/1000:,.1f}k' if value >= 1000 else f'{value:,.0f}'
def apply_axes_style(ax, y_grid=True):
    ax.spines['left'].set_color(LINE)
    ax.spines['bottom'].set_color(LINE)
    ax.tick_params(length=0)
    if y_grid:
        ax.grid(axis='y', color=GRID, linewidth=0.8)
    else:
        ax.grid(axis='x', color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
def add_title_block(ax, title, subtitle):
    ax.set_title(title, loc='left', pad=20)
    ax.text(0, 1.03, subtitle, transform=ax.transAxes, fontsize=10.5, color=MID, ha='left')
def add_caption(fig, text):
    fig.text(0.01, 0.01, text, ha='left', va='bottom', fontsize=9, color=MID)
def add_figure_title_block(fig, title, subtitle):
    fig.text(0.01, 0.965, title, ha='left', va='top', fontsize=18, fontweight='bold', color=DARK)
    fig.text(0.01, 0.928, subtitle, ha='left', va='top', fontsize=10.5, color=MID)
def make_table_1_png(q1_table, output_path=CHART_DIR / 'table_1_tenure_by_ami_status.png'):
    table_df = q1_table.pivot(index='tenure_simple', columns='ami_group', values='weighted_households').fillna(0)
    table_df = table_df.loc[['Renter', 'Owner']].reset_index().rename(columns={'tenure_simple': 'Tenure'})
    table_df['Below 60% AMI_label'] = table_df['Below 60% AMI'].map(lambda x: f'{x:,.0f}')
    table_df['At/Above 60% AMI_label'] = table_df['At/Above 60% AMI'].map(lambda x: f'{x:,.0f}')
    table_df['y'] = [1, 0]

    left = alt.Chart(table_df).mark_text(align='left', font=ALT_FONT, fontSize=12, color=DARK).encode(
        y=alt.Y('y:O', axis=None),
        text='Tenure:N'
    ).properties(width=120, height=90, title='Tenure')

    mid = alt.Chart(table_df).mark_text(align='right', font=ALT_FONT, fontSize=12, color=DARK).encode(
        y=alt.Y('y:O', axis=None),
        text='Below 60% AMI_label:N'
    ).properties(width=150, height=90, title='Below 60% AMI')

    right = alt.Chart(table_df).mark_text(align='right', font=ALT_FONT, fontSize=12, color=DARK).encode(
        y=alt.Y('y:O', axis=None),
        text='At/Above 60% AMI_label:N'
    ).properties(width=170, height=90, title='At/Above 60% AMI')

    chart = alt.hconcat(left, mid, right, spacing=28).properties(
        title=alt.TitleParams(
            text='Households by tenure and 60% of AMI status',
            subtitle=['Weighted Dallas household estimates using the supplied ACS / PUMS extract.'],
        ),
        padding={'left': 12, 'right': 18, 'top': 12, 'bottom': 10},
    )
    chart = apply_altair_theme(chart)
    save_altair_chart(chart, output_path)
    return chart
def save_altair_chart(chart, output_path, scale_factor=3):
    output_path = Path(output_path)
    try:
        chart.save(output_path, scale_factor=scale_factor)
    except Exception as exc:
        html_path = output_path.with_suffix('.html')
        chart.save(html_path)
        print(f'PNG export failed ({exc}). Saved HTML instead to {html_path}. Install vl-convert-python for PNG export.')

def altair_base(chart, width=None, height=None):
    if width is not None or height is not None:
        chart = chart.properties(width=width, height=height)
    return chart

def apply_altair_theme(chart):
    return chart.configure_view(stroke=None).configure_title(
        anchor='start',
        font=ALT_FONT,
        fontSize=20,
        subtitleFont=ALT_FONT,
        subtitleFontSize=12,
        subtitleColor=MID,
        offset=18,
    ).configure_axis(
        domainColor=LINE,
        domainWidth=1,
        gridColor=GRID,
        gridWidth=1,
        labelColor=MID,
        labelFont=ALT_FONT,
        labelFontSize=12,
        titleColor=MID,
        titleFont=ALT_FONT,
        titleFontSize=12,
        tickColor=LINE,
        tickSize=3,
    ).configure_axisY(
        labelLimit=220,
    ).configure_legend(
        labelFont=ALT_FONT,
        titleFont=ALT_FONT,
        orient='top',
        direction='horizontal',
        symbolType='square',
        padding=6,
    ).configure(background=BG)

def make_chart_1(q1_table, output_path=CHART_DIR / 'chart_1_ami_by_tenure.png'):
    chart_df = q1_table.copy()
    chart_df['tenure_simple'] = pd.Categorical(chart_df['tenure_simple'], categories=['Renter', 'Owner'], ordered=True)
    chart_df['ami_group'] = pd.Categorical(chart_df['ami_group'], categories=['Below 60% AMI', 'At/Above 60% AMI'], ordered=True)
    color_scale = alt.Scale(domain=['Below 60% AMI', 'At/Above 60% AMI'], range=['#2E86AB', '#D9E3EA'])

    bars = alt.Chart(chart_df).mark_bar(size=26).encode(
        x=alt.X('tenure_simple:N', title=None, sort=['Renter', 'Owner']),
        y=alt.Y('weighted_households:Q', title='Weighted Dallas households', axis=alt.Axis(format='~s', grid=True)),
        xOffset=alt.XOffset('ami_group:N', sort=['Below 60% AMI', 'At/Above 60% AMI']),
        color=alt.Color('ami_group:N', title=None, scale=color_scale),
    )

    labels = alt.Chart(chart_df).mark_text(dy=-8, font=ALT_FONT, fontSize=11, color=DARK).encode(
        x=alt.X('tenure_simple:N', sort=['Renter', 'Owner']),
        y=alt.Y('weighted_households:Q'),
        xOffset=alt.XOffset('ami_group:N', sort=['Below 60% AMI', 'At/Above 60% AMI']),
        text=alt.Text('weighted_households:Q', format=',.0f'),
    )

    chart = (bars + labels).properties(
            title=alt.TitleParams(
                text='How many Dallas households fall below 60% of AMI, and how does that differ by tenure?',
                subtitle=['Renters make up the larger share of households below 60% of AMI.'],
            ),
            width=360,
            height=310,
            padding={'left': 12, 'right': 18, 'top': 10, 'bottom': 10},
        )
    chart = apply_altair_theme(chart)
    save_altair_chart(chart, output_path)
    return chart

def make_chart_2(race_comparison, output_path=CHART_DIR / 'chart_2_target_vs_all_race.png'):
    chart_df = race_comparison.copy().sort_values('share_target', ascending=False).reset_index(drop=True)
    category_order = chart_df['category'].tolist()
    base = chart_df.copy()

    left = alt.Chart(base).mark_bar(size=24, color='#DCE5EC').encode(
        y=alt.Y('category:N', sort=category_order, title='Race / ethnicity', axis=alt.Axis(labelLimit=220)),
        x=alt.X('share_all:Q', title='Share of all Dallas households', axis=alt.Axis(format='.0%', grid=True)),
    )
    left_labels = alt.Chart(base).mark_text(align='left', dx=6, font=ALT_FONT, fontSize=11, color=DARK).encode(
        y=alt.Y('category:N', sort=category_order),
        x=alt.X('share_all:Q'),
        text=alt.Text('share_all:Q', format='.0%'),
    )
    left_chart = altair_base((left + left_labels).properties(width=260, height=320, title='All households'))

    right = alt.Chart(base).mark_bar(size=24, color='#F28E2B').encode(
        y=alt.Y('category:N', sort=category_order, title=None, axis=alt.Axis(labels=False, ticks=False, domain=False)),
        x=alt.X('share_target:Q', title='Share of cost-burdened households', axis=alt.Axis(format='.0%', grid=True)),
    )
    right_labels = alt.Chart(base).mark_text(align='left', dx=6, font=ALT_FONT, fontSize=11, color=DARK).encode(
        y=alt.Y('category:N', sort=category_order),
        x=alt.X('share_target:Q'),
        text=alt.Text('share_target:Q', format='.0%'),
    )
    right_chart = altair_base((right + right_labels).properties(width=260, height=320, title='Low-income, cost-burdened'))

    chart = alt.hconcat(left_chart, right_chart, spacing=42).resolve_scale(y='shared').properties(
        title=alt.TitleParams(
            text='Who makes up the low-income, cost-burdened population?',
            subtitle=['The burdened cohort differs from Dallas households overall.'],
        ),
        padding={'left': 12, 'right': 18, 'top': 12, 'bottom': 10},
    )
    chart = apply_altair_theme(chart)
    save_altair_chart(chart, output_path)
    return chart

def make_chart_3(df, chart_variable, chart_label, output_path=CHART_DIR / 'chart_3_burden_rates_indicator.png'):
    subset = df.loc[df['below_60_ami'] & (~df['non_positive_income_flag']) & df['annual_housing_cost'].notna()].copy()

    if chart_variable == 'Tenure':
        chart_df = burden_rate_within_subset(df, 'Tenure', df['below_60_ami']).copy()
        chart_df['label'] = chart_df['Tenure'].str.replace(' Household', '', regex=False)
        bars = alt.Chart(chart_df).mark_bar(size=26, color='#59A14F').encode(
            y=alt.Y('label:N', sort='-x', title='Tenure'),
            x=alt.X('burden_rate:Q', title='Percentage of households that are cost burdened', axis=alt.Axis(format='.0%', grid=True)),
        )
        labels = alt.Chart(chart_df).mark_text(align='left', dx=6, font=ALT_FONT, fontSize=11, color=DARK).encode(
            y=alt.Y('label:N', sort='-x'),
            x=alt.X('burden_rate:Q'),
            text=alt.Text('burden_rate:Q', format='.0%'),
        )
        chart = (bars + labels).properties(
            title=alt.TitleParams(
                text='Which low-income households are most likely to be cost burdened?',
                subtitle=['Rates are shown within the below-60%-AMI population.'],
            ),
            width=480,
            height=180,
            padding={'left': 12, 'right': 18, 'top': 12, 'bottom': 10},
        )
        chart = apply_altair_theme(chart)
        save_altair_chart(chart, output_path)
        return chart

    chart_df = subset.groupby(['Tenure', chart_variable], dropna=False).apply(
        lambda g: pd.Series({
            'weighted_households': g['WGTP'].sum(),
            'weighted_burdened': g.loc[g['cost_burdened'], 'WGTP'].sum(),
        })
    ).reset_index()
    chart_df['burden_rate'] = chart_df['weighted_burdened'] / chart_df['weighted_households']
    chart_df = chart_df[chart_df['weighted_households'] > 0].copy()
    chart_df['tenure_simple'] = chart_df['Tenure'].str.replace(' Household', '', regex=False)
    chart_df['label'] = chart_df[chart_variable].astype(str)
    category_order = chart_df.groupby('label')['weighted_households'].sum().sort_values(ascending=False).index.tolist()

    left_df = chart_df.loc[chart_df['tenure_simple'] == 'Renter'].copy()
    right_df = chart_df.loc[chart_df['tenure_simple'] == 'Owner'].copy()

    left = alt.Chart(left_df).mark_bar(size=24, color='#59A14F').encode(
        y=alt.Y('label:N', sort=category_order, title=chart_label, axis=alt.Axis(labelLimit=220)),
        x=alt.X('burden_rate:Q', title='Percentage of households that are cost burdened', axis=alt.Axis(format='.0%', grid=True)),
    )
    left_labels = alt.Chart(left_df).mark_text(align='left', dx=6, font=ALT_FONT, fontSize=11, color=DARK).encode(
        y=alt.Y('label:N', sort=category_order),
        x=alt.X('burden_rate:Q'),
        text=alt.Text('burden_rate:Q', format='.0%'),
    )
    left_chart = altair_base((left + left_labels).properties(width=260, height=320, title='Renter'))

    right = alt.Chart(right_df).mark_bar(size=24, color='#A0CBE8').encode(
        y=alt.Y('label:N', sort=category_order, title=None, axis=alt.Axis(labels=False, ticks=False, domain=False)),
        x=alt.X('burden_rate:Q', title='Percentage of households that are cost burdened', axis=alt.Axis(format='.0%', grid=True)),
    )
    right_labels = alt.Chart(right_df).mark_text(align='left', dx=6, font=ALT_FONT, fontSize=11, color=DARK).encode(
        y=alt.Y('label:N', sort=category_order),
        x=alt.X('burden_rate:Q'),
        text=alt.Text('burden_rate:Q', format='.0%'),
    )
    right_chart = altair_base((right + right_labels).properties(width=260, height=320, title='Owner'))

    chart = alt.hconcat(left_chart, right_chart, spacing=42).resolve_scale(y='shared').properties(
        title=alt.TitleParams(
            text='Which low-income households are most likely to be cost burdened?',
            subtitle=[f'Rates are shown within the below-60%-AMI population by tenure and {chart_label.lower()}.'],
        ),
        padding={'left': 12, 'right': 18, 'top': 12, 'bottom': 10},
    )
    chart = apply_altair_theme(chart)
    save_altair_chart(chart, output_path)
    return chart

def fetch_puma_context(df, year=2021, state_fips='48'):
    # Lightweight extension: share of households with no vehicle available by PUMA.
    url = f'https://api.census.gov/data/{year}/acs/acs5'
    params = {
        'get': 'NAME,B08201_001E,B08201_002E',
        'for': 'public use microdata area:*',
        'in': f'state:{state_fips}',
    }
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()
        context = pd.DataFrame(payload[1:], columns=payload[0])
        context[['B08201_001E', 'B08201_002E']] = context[['B08201_001E', 'B08201_002E']].apply(pd.to_numeric)
        context['puma5'] = context['state'] + context['public use microdata area']
        context['no_vehicle_share'] = context['B08201_002E'] / context['B08201_001E']
        context = context[['puma5', 'NAME', 'no_vehicle_share']]
        merged = df.merge(context, on='puma5', how='left')
        return merged, context
    except Exception as exc:
        print(f'Context fetch skipped: {exc}')
        return df.copy(), pd.DataFrame()
def build_vehicle_access_context_summary(df):
    if 'no_vehicle_share' not in df.columns or df['no_vehicle_share'].isna().all():
        return pd.DataFrame()
    summary = df.copy()
    median_share = summary['no_vehicle_share'].median()
    summary['vehicle_access_context'] = pd.Series(pd.NA, index=summary.index, dtype='object')
    summary.loc[summary['no_vehicle_share'].notna(), 'vehicle_access_context'] = 'Higher no-vehicle PUMAs'
    summary.loc[summary['no_vehicle_share'].notna() & (summary['no_vehicle_share'] < median_share), 'vehicle_access_context'] = 'Lower no-vehicle PUMAs'
    grouped = summary.loc[summary['vehicle_access_context'].notna()].groupby('vehicle_access_context').apply(
        lambda g: pd.Series({
            'weighted_households_all': g['WGTP'].sum(),
            'weighted_households_target': g.loc[g['target_group'], 'WGTP'].sum(),
            'mean_no_vehicle_share': g['no_vehicle_share'].mean(),
        })
    ).reset_index()
    grouped['share_all_households'] = grouped['weighted_households_all'] / grouped['weighted_households_all'].sum()
    grouped['share_target_households'] = grouped['weighted_households_target'] / grouped['weighted_households_target'].sum()
    grouped['target_minus_all_gap'] = grouped['share_target_households'] - grouped['share_all_households']
    return grouped.sort_values('target_minus_all_gap', ascending=False)
def build_memo_markdown(results):
    q1 = results['q1_table'].pivot(index='tenure_simple', columns='ami_group', values='weighted_households').fillna(0)
    target_households = results['target_estimate']['weighted_households']
    target_share = results['target_estimate']['share_of_all_households']
    below_share = results['target_estimate']['share_of_below_60_ami_households']
    race = results['race_comparison'].set_index('category')
    vehicle_context = results.get('vehicle_access_context_summary', pd.DataFrame())

    vehicle_context_line = ''
    if not vehicle_context.empty:
        top_context = vehicle_context.iloc[0]
        vehicle_context_line = f"- The target group is somewhat more concentrated in {top_context['vehicle_access_context'].lower()}, where average no-vehicle rates are approximately {top_context['mean_no_vehicle_share']:.1%}. This suggests that housing affordability and transportation disadvantage may overlap for some Dallas households, though further analysis would be needed to confirm that relationship."

    memo = f"""
**Dallas Housing Needs Assessment — Summary Findings and Policy Implications**
    
# Approach
This analysis uses the provided Dallas household extract derived from ACS / PUMS records, applying household weights (`WGTP`) throughout. Households are classified as below 60% of AMI using Dallas 2020 HUD income limits by household size. Housing cost burden is defined descriptively as annualized renter or owner housing cost divided by inflation-adjusted household income, with burden flagged when that ratio exceeds 30%.

# Key Findings
- Approximately {q1.loc['Renter', 'Below 60% AMI'] + q1.loc['Owner', 'Below 60% AMI']:,.0f} weighted Dallas households fall below 60% of AMI.
- Approximately {target_households:,.0f} households are both below 60% of AMI and cost burdened, representing {target_share:.1%} of all households.
- Within the below-60%-AMI population, {below_share:.1%} of households are cost burdened under this measure.
- The low-income population is renter-dominated: {q1.loc['Renter', 'Below 60% AMI']:,.0f} renter households fall below 60% of AMI, compared with {q1.loc['Owner', 'Below 60% AMI']:,.0f} owner households.
- Compared with all Dallas households, the cost-burdened low-income population includes a larger share of Black and Hispanic households and a smaller share of White households.
{vehicle_context_line}

# Which households appear most vulnerable to cost burden?
The strongest descriptive indicator is tenure: within the below-60%-AMI population, renters are substantially more likely to be cost burdened than owners. Among additional household characteristics, housing typology shows one of the clearest burden gradients, with higher burden rates in multifamily structures than in single-family homes. The target group is also more concentrated among one-person households and single-parent households relative to the Dallas household population overall.

These patterns are descriptive but consistent with structural differences in housing access, tenure, and household composition across the city.

# Recommended setup for next-step analysis
A next-step model should be a weighted logistic regression restricted to households below 60% of AMI, with `cost_burdened` as the dependent variable. Predictor variables should include tenure, race / ethnicity, household size, single-parent status, senior presence, recent move status, and housing typology. The objective should be interpretability and policy relevance rather than predictive optimization.

# Possible Policy Implications for Dallas
- Observation: Low-income renters make up the largest share of cost-burdened households.  
  Interpretation: Affordability pressure is concentrated in the rental market.  
  Implication: Prioritize rental assistance, eviction prevention, and preservation of naturally occurring affordable rental housing.

- Observation: Higher burden rates are observed in multifamily housing typologies.  
  Interpretation: Many cost-burdened households are already located in rental-oriented building types.  
  Implication: Target preservation and stabilization strategies toward multifamily properties serving lower-income households.

- Observation: Racial disparities are evident within the cost-burdened population.  
  Interpretation: Housing cost burden is not evenly distributed across Dallas households.  
  Implication: Incorporate equity-focused targeting into program design, outreach, and investment strategies.

- Observation: One-person and single-parent households are overrepresented among cost-burdened households.  
  Interpretation: Household composition plays a meaningful role in shaping affordability constraints.  
  Implication: Expand support for both smaller low-cost units and family-sized affordable housing, alongside targeted assistance for households with children.

- Observation: Where included, higher no-vehicle rates suggest overlapping housing and mobility constraints.  
  Interpretation: Some households may face compounded access and affordability challenges.  
  Implication: Coordinate housing interventions with transportation and access planning in affected areas.

# Limitations
- AMI thresholds reflect an earlier income-limit vintage than the ACS / PUMS data; aligning vintages would improve accuracy.
- PUMS is a sample, so results are estimates.
- PUMA geography approximates, but does not perfectly match, Dallas city boundaries.
- Renter and owner housing cost measures are not directly comparable.
- Zero or negative income cases are treated as missing for burden calculations rather than forced into extreme ratios.
- Race / ethnicity is based on the household reference person and does not capture full household composition.
- `SERIALNO` does not function as a reliable unique household identifier in this extract, limiting certain linkage checks.

# Optional next steps for workflow / dashboard development
The analysis is structured as a modular workflow separating income classification, cost burden calculation, summary outputs, and chart generation. This structure supports reuse across ACS releases, adaptation to other geographies, and integration into a dashboard-based analytical tool.
""".strip()
    return memo
def run_dallas_housing_assessment():
    loaded = load_input_data()
    df = inspect_and_standardize_columns(loaded['data'])
    df = assign_ami_thresholds(df)
    df = calculate_housing_cost_burden(df)
    df = add_analysis_fields(df)
    df_with_context, context_df = fetch_puma_context(df)
    q1_table = build_weighted_summary(df_with_context, ['tenure_simple', 'ami_group'])
    target_estimate = {
        'weighted_households': df_with_context.loc[df_with_context['target_group'], 'WGTP'].sum(),
        'share_of_all_households': df_with_context.loc[df_with_context['target_group'], 'WGTP'].sum() / df_with_context['WGTP'].sum(),
        'share_of_below_60_ami_households': df_with_context.loc[df_with_context['target_group'], 'WGTP'].sum() / df_with_context.loc[df_with_context['below_60_ami'], 'WGTP'].sum(),
    }
    comparison_table = compare_target_group(df_with_context)
    race_comparison = summarize_distribution(df_with_context, 'race_ethnicity')
    indicator_ranking = identify_indicator_variable(df_with_context)
    chart_3_indicator = select_chart_3_variable(indicator_ranking)
    candidate_variables_table = pd.DataFrame(CANDIDATE_VARIABLES).T.rename_axis('variable_key').reset_index()
    validation_check = build_validation_check(df_with_context)
    vehicle_access_context_summary = build_vehicle_access_context_summary(df_with_context)
    q1_table.to_csv(TABLE_DIR / 'table_1_tenure_by_ami_status.csv', index=False)
    comparison_table.to_csv(TABLE_DIR / 'table_2_target_group_comparison.csv', index=False)
    indicator_ranking.to_csv(TABLE_DIR / 'table_3_indicator_ranking.csv', index=False)
    if not validation_check.empty:
        validation_check.to_csv(TABLE_DIR / 'table_4_validation_against_supplied_burden.csv', index=False)
    if not context_df.empty:
        context_df.to_csv(TABLE_DIR / 'table_5_puma_context_no_vehicle.csv', index=False)
    if not vehicle_access_context_summary.empty:
        vehicle_access_context_summary.to_csv(TABLE_DIR / 'table_6_vehicle_access_context_summary.csv', index=False)
    make_table_1_png(q1_table)
    make_chart_1(q1_table)
    make_chart_2(race_comparison)
    make_chart_3(df_with_context, chart_3_indicator['variable'], chart_3_indicator['label'])
    memo_markdown = build_memo_markdown({
        'q1_table': q1_table,
        'target_estimate': target_estimate,
        'race_comparison': race_comparison,
        'vehicle_access_context_summary': vehicle_access_context_summary,
    })
    (REPORT_DIR / 'dallas_housing_assessment_memo.md').write_text(memo_markdown)
    return {
        'data': df_with_context,
        'glossary': loaded['glossary'],
        'q1_table': q1_table,
        'target_estimate': target_estimate,
        'comparison_table': comparison_table,
        'race_comparison': race_comparison,
        'indicator_ranking': indicator_ranking,
        'candidate_variables_table': candidate_variables_table,
        'validation_check': validation_check,
        'context_table': context_df,
        'vehicle_access_context_summary': vehicle_access_context_summary,
        'chart_3_indicator': chart_3_indicator,
        'memo_markdown': memo_markdown,
    }


## 4. Results

This section produces the required household estimates, comparison tables, and validation checks used in the final interpretation.


In [127]:
results = run_dallas_housing_assessment()
display(Markdown('### Tenure by 60% AMI status'))
display(results['q1_table'].pivot(index='tenure_simple', columns='ami_group', values='weighted_households').fillna(0).round(0).astype(int))


### Tenure by 60% AMI status

ami_group,At/Above 60% AMI,Below 60% AMI
tenure_simple,,
Owner,185121,62847
Renter,151556,120974


In [128]:
display(Markdown('### Low-income, cost-burdened households'))
display(pd.DataFrame([results['target_estimate']]).assign(
    weighted_households=lambda d: d['weighted_households'].round(0).astype(int),
    share_of_all_households=lambda d: d['share_of_all_households'].map('{:.1%}'.format),
    share_of_below_60_ami_households=lambda d: d['share_of_below_60_ami_households'].map('{:.1%}'.format),
))
display(Markdown('### Descriptive comparisons'))
display(results['comparison_table'].copy())
display(Markdown('### Candidate variables for next-step modeling'))
display(results['candidate_variables_table'][['label', 'level', 'why_it_matters', 'summary_method', 'model_ready']])
display(Markdown('### Validation against supplied burden fields'))
display(results['validation_check'].assign(
    mean_abs_difference_pct_points=lambda d: d['mean_abs_difference_pct_points'].round(2),
    burden_flag_agreement_rate=lambda d: d['burden_flag_agreement_rate'].map('{:.1%}'.format),
))
display(Markdown('Derived burden measures are used for the analysis; supplied burden fields are shown only as a check.'))


### Low-income, cost-burdened households

,weighted_households,share_of_all_households,share_of_below_60_ami_households
0,137494,26.4%,74.8%


### Descriptive comparisons

,variable,category,weighted_households_all,weighted_households_target,share_all,share_target,share_point_difference
0,Tenure,Renter Household,272530,100810,0.523595,0.733196,0.209601
1,Tenure,Owner Household,247968,36684,0.476405,0.266804,-0.209601
2,Race / ethnicity of reference person,Hispanic,170706,52080,0.327967,0.378780,0.050813
3,Race / ethnicity of reference person,Black,123544,49756,0.237357,0.361878,0.124520
4,Race / ethnicity of reference person,White,198545,29669,0.381452,0.215784,-0.165668
5,Race / ethnicity of reference person,Asian,16796,3426,0.032269,0.024917,-0.007352
6,Race / ethnicity of reference person,Other,10907,2563,0.020955,0.018641,-0.002314
7,Household size,1 person,168498,59205,0.323725,0.430601,0.106876
8,Household size,3-4 persons,135204,33006,0.259759,0.240054,-0.019705
9,Household size,2 persons,151458,27551,0.290987,0.200380,-0.090607


### Candidate variables for next-step modeling

,label,level,why_it_matters,summary_method,model_ready
0,Tenure,Household,Tenure captures different affordability mechan...,Weighted counts and burden rates by renter ver...,Yes
1,Race / ethnicity of reference person,Reference person / household proxy,Racial and ethnic disparities often reflect lo...,Weighted population shares for all households ...,Yes
2,Household size,Household,Household size affects both AMI thresholds and...,Weighted shares and burden rates by size band.,Yes
3,Single-parent household,Household,Single-parent households often have tighter bu...,"Burden rates within the below-60%-AMI subset, ...",Yes
4,Senior present,Household,Senior households may face fixed-income constr...,Burden rates and target-group shares.,Yes
5,Structure type / housing typology,Household,Housing form provides context for where afford...,Weighted shares and burden rates by typology.,Yes
6,Moved in last 24 months,Household,Recent movers may be more exposed to current m...,Burden rates and target-group shares by recent...,Yes


### Validation against supplied burden fields

,rows_compared_on_burden_rate,mean_abs_difference_pct_points,rows_compared_on_burden_flag,burden_flag_agreement_rate
0,23945,0.0,23945,100.0%


Derived burden measures are used for the analysis; supplied burden fields are shown only as a check.

## 5. Indicator Analysis

This section compares candidate household characteristics within the below-60%-AMI population to identify the strongest descriptive indicator for Chart 3.


In [129]:
display(Markdown('### Indicator scan within below-60%-AMI households'))
display(results['indicator_ranking'].assign(spread=lambda d: d['spread'].map('{:.1%}'.format)))
display(Markdown(f"Chart 3 uses `{results['chart_3_indicator']['label']}`."))


### Indicator scan within below-60%-AMI households

,variable,label,spread,use_for_chart_3
0,Tenure,Tenure,27.4%,False
5,typology,Housing typology,21.8%,True
4,moved_label,Recent move status,16.2%,True
3,senior_label,Senior presence,14.5%,True
1,single_parent_label,Single-parent household,11.8%,True
2,household_size_group,Household size,11.7%,True


Chart 3 uses `Housing typology`.

## 6. Build Charts

This section renders the three memo-ready figures that summarize scale, demographic composition, and variation in cost burden.


In [130]:
chart1 = make_chart_1(results['q1_table'])
display(chart1)

chart2 = make_chart_2(results['race_comparison'])
display(chart2)

chart3 = make_chart_3(
    results['data'],
    results['chart_3_indicator']['variable'],
    results['chart_3_indicator']['label'],
)
display(chart3)


alt.LayerChart(...)

alt.HConcatChart(...)

alt.HConcatChart(...)

## 7. PUMA context

This section adds one lightweight ACS context measure at the PUMA level to support housing-plus-mobility interpretation.


In [131]:
if not results['context_table'].empty:
    display(Markdown('### PUMA context: no vehicle available'))
    display(results['context_table'].sort_values('no_vehicle_share', ascending=False).head(12))
if not results['vehicle_access_context_summary'].empty:
    display(Markdown('### Housing and mobility context'))
    display(results['vehicle_access_context_summary'].assign(
        mean_no_vehicle_share=lambda d: d['mean_no_vehicle_share'].map('{:.1%}'.format),
        share_all_households=lambda d: d['share_all_households'].map('{:.1%}'.format),
        share_target_households=lambda d: d['share_target_households'].map('{:.1%}'.format),
        target_minus_all_gap=lambda d: d['target_minus_all_gap'].map('{:+.1%}'.format),
    ))
display(Markdown('`No vehicle available` is included as a single lightweight ACS context measure at the PUMA level.'))


### PUMA context: no vehicle available

,puma5,NAME,no_vehicle_share
175,4805901,"San Antonio City (Central) PUMA, Texas",0.181416
85,4803304,"El Paso City (Central) PUMA, Texas",0.173919
49,4802314,Dallas City (East Central)--South of I-30 & Ea...,0.163650
120,4804614,"Houston City (West)--Westpark Tollway, Between...",0.162713
109,4804603,"Houston City (South Central)--South of US-59, ...",0.138301
177,4805903,San Antonio City (Southeast)--Inside Loop I-41...,0.137884
108,4804602,Houston City (East Central)--East of I-45 & In...,0.125857
51,4802316,Dallas City (South Central)--North of I-20 & W...,0.122947
178,4805904,San Antonio City (Northwest)--Inside Loop I-41...,0.114606
121,4804615,Houston (Southwest) & Bellaire (Southeast) Cit...,0.110016


### Housing and mobility context

,vehicle_access_context,weighted_households_all,weighted_households_target,mean_no_vehicle_share,share_all_households,share_target_households,target_minus_all_gap
0,Higher no-vehicle PUMAs,316254.0,88573.0,9.8%,60.8%,64.4%,+3.7%
1,Lower no-vehicle PUMAs,204244.0,48921.0,6.1%,39.2%,35.6%,-3.7%


`No vehicle available` is included as a single lightweight ACS context measure at the PUMA level.

## 8. Memo Text

This section prints the memo-ready narrative generated from the current results and writes the same text to `outputs/report/dallas_housing_assessment_memo.md`.


In [132]:
display(Markdown('### Memo-ready draft'))
display(Markdown(results['memo_markdown']))
print(REPORT_DIR / 'dallas_housing_assessment_memo.md')


**Dallas Housing Needs Assessment — Summary Findings and Policy Implications**

# Approach
This analysis uses the provided Dallas household extract derived from ACS / PUMS records, applying household weights (`WGTP`) throughout. Households are classified as below 60% of AMI using Dallas 2020 HUD income limits by household size. Housing cost burden is defined descriptively as annualized renter or owner housing cost divided by inflation-adjusted household income, with burden flagged when that ratio exceeds 30%.

# Key Findings
- Approximately 183,821 weighted Dallas households fall below 60% of AMI.
- Approximately 137,494 households are both below 60% of AMI and cost burdened, representing 26.4% of all households.
- Within the below-60%-AMI population, 74.8% of households are cost burdened under this measure.
- The low-income population is renter-dominated: 120,974 renter households fall below 60% of AMI, compared with 62,847 owner households.
- Compared with all Dallas households, the cost-burdened low-income population includes a larger share of Black and Hispanic households and a smaller share of White households.
- The target group is somewhat more concentrated in higher no-vehicle pumas, where average no-vehicle rates are approximately 9.8%. This suggests that housing affordability and transportation disadvantage may overlap for some Dallas households, though further analysis would be needed to confirm that relationship.

# Which households appear most vulnerable to cost burden?
The strongest descriptive indicator is tenure: within the below-60%-AMI population, renters are substantially more likely to be cost burdened than owners. Among additional household characteristics, housing typology shows one of the clearest burden gradients, with higher burden rates in multifamily structures than in single-family homes. The target group is also more concentrated among one-person households and single-parent households relative to the Dallas household population overall.

These patterns are descriptive but consistent with structural differences in housing access, tenure, and household composition across the city.

# Recommended setup for next-step analysis
A next-step model should be a weighted logistic regression restricted to households below 60% of AMI, with `cost_burdened` as the dependent variable. Predictor variables should include tenure, race / ethnicity, household size, single-parent status, senior presence, recent move status, and housing typology. The objective should be interpretability and policy relevance rather than predictive optimization.

# Possible Policy Implications for Dallas
- Observation: Low-income renters make up the largest share of cost-burdened households.  
  Interpretation: Affordability pressure is concentrated in the rental market.  
  Implication: Prioritize rental assistance, eviction prevention, and preservation of naturally occurring affordable rental housing.

- Observation: Higher burden rates are observed in multifamily housing typologies.  
  Interpretation: Many cost-burdened households are already located in rental-oriented building types.  
  Implication: Target preservation and stabilization strategies toward multifamily properties serving lower-income households.

- Observation: Racial disparities are evident within the cost-burdened population.  
  Interpretation: Housing cost burden is not evenly distributed across Dallas households.  
  Implication: Incorporate equity-focused targeting into program design, outreach, and investment strategies.

- Observation: One-person and single-parent households are overrepresented among cost-burdened households.  
  Interpretation: Household composition plays a meaningful role in shaping affordability constraints.  
  Implication: Expand support for both smaller low-cost units and family-sized affordable housing, alongside targeted assistance for households with children.

- Observation: Where included, higher no-vehicle rates suggest overlapping housing and mobility constraints.  
  Interpretation: Some households may face compounded access and affordability challenges.  
  Implication: Coordinate housing interventions with transportation and access planning in affected areas.

# Limitations
- AMI thresholds reflect an earlier income-limit vintage than the ACS / PUMS data; aligning vintages would improve accuracy.
- PUMS is a sample, so results are estimates.
- PUMA geography approximates, but does not perfectly match, Dallas city boundaries.
- Renter and owner housing cost measures are not directly comparable.
- Zero or negative income cases are treated as missing for burden calculations rather than forced into extreme ratios.
- Race / ethnicity is based on the household reference person and does not capture full household composition.
- `SERIALNO` does not function as a reliable unique household identifier in this extract, limiting certain linkage checks.

# Optional next steps for workflow / dashboard development
The analysis is structured as a modular workflow separating income classification, cost burden calculation, summary outputs, and chart generation. This structure supports reuse across ACS releases, adaptation to other geographies, and integration into a dashboard-based analytical tool.